# Gyroids_STL_class

**What this notebook showcases in `gyroid_utils`:** the same TPMS/gyroid generation use case as `Gyroids_STL.ipynb`, but using the object-oriented `GyroidModel` class (`gyroid.py`) instead of standalone functions. It walks through defining the design-space grid, instantiating gyroid models with the `'abs'`, `'signed'`, and `'distance'` field modes, and generating/exporting the resulting surface meshes.


In [1]:
import numpy as np
from stl import mesh as stl_mesh
import plotly.graph_objects as go  # for visualization
import os
import trimesh


import gyroid_utils
from gyroid_utils.utils import reload_all
reload_all()

working_path = os.getcwd()
print("Current working directory:", working_path)




[gyroid_utils] version 3.1.4 loaded
[gyroid_utils] version 3.1.4 loaded
gyroid_utils: all modules reloaded
Current working directory: c:\Users\cofo\Documents\02 - GitHub\GYROIDS\notebooks


## 1 — Define the Design Space

Set up the 3D coordinate grid that the gyroid scalar field will be evaluated on.

- **Domain size**: 100 × 100 × 100 units along *x*, *y*, and *z*.
- **Grid spacing** (`dx_grid`, `dy_grid`, `dz_grid`): computed as `domain_size / 100`, giving one sample per unit (i.e. 101 nodes per axis including the endpoint).

The total number of voxels is printed as a sanity check before proceeding.

In [2]:
# size of domain
pz2 = 100
py2 = 100
px2 = 100

# --- Discretization of the domain ---
# Resolution in each axis, calculated as a funcion of the size of the gyroid's unit cell
dx_grid = px2 / 100
dy_grid = py2 / 100
dz_grid = pz2 / 100

# 1D coordinate arrays. np.arange(stop + step, step) includes the endpoint like MATLAB's colon with step.
x1 = np.arange(0, px2 + dx_grid, dx_grid)       # x positions from 0 to pz2 + dx_grid, and the step size is dx_grid
y1 = np.arange(0, py2 + dy_grid, dy_grid)       # y positions from 0 to py2, step dy_grid
z1 = np.arange(0, pz2 + dz_grid, dz_grid)       # z positions from 0 to pz2, step dz_grid

# Create 3D coordinate grids. indexing='ij' -> (X,Y,Z) follow x1,y1,z1 order like MATLAB.
x, y, z = np.meshgrid(x1, y1, z1, indexing='ij')

print(f"x-axis resolution {np.size(x1)=}, y-axis resolution {np.size(y1)=}, z-axis resolution {np.size(z1)=}")
print(f"In total, {np.size(x)} voxels in the 3D grid")

x-axis resolution np.size(x1)=101, y-axis resolution np.size(y1)=101, z-axis resolution np.size(z1)=101
In total, 1030301 voxels in the 3D grid


## 2 — Generate Gyroid Models (three field modes)

The next three cells each instantiate a `GyroidModel` and run the same pipeline with a different scalar-field computation mode. All three share the same workflow:

1. **Set the unit-cell period** — `px`, `py`, `pz` showcase 3 different way to build variable:
   - **a.** as a matrice, changing in space
   - **b.** as a matrice, constant in space
   - **c.** as a single constant (float)
2. **Set the iso-value offset `t`** — controls wall thickness / offset from the mid-surface. It can be built the same way 'px' is built.
3. **Instantiate `GyroidModel`** — passes the grid and period arrays to the constructor.
4. **`compute_field(mode=...)`** — evaluates the TPMS function across the grid:
   - `"abs"` (`t = 0.1`): absolute value of the gyroid function; produces a thin symmetric shell around the mid-surface.
   - `"signed"` (`t = 0.5`): signed (raw) field; suitable for defining solid/void regions by sign.
   - `"distance"` (`t = 4`): approximate distance field; the iso-value has physical length units, giving direct thickness control.
5. **`twod_view_of_matrix`** — renders a 2-D slice of the scalar field for a visual sanity-check.
6. **`generate_mesh()`** — extracts the iso-surface via Marching Cubes.
7. **`simplify_mesh(target_faces=10000)`** — reduces triangle count for manageable file sizes.
8. **`check_mesh_quality()`** — reports mesh statistics (watertightness, degenerate faces, etc.).
9. **Save** — writes a preview 3D as `.html`, a `.npz` checkpoint, and an `.stl` export.

In [ ]:
# =============================================
# ============== abs gyroids ==================
# =============================================
file_name = "gyroid-test-abs"

# ---  Define Period of gyroid unit cell -------
px = np.copy(x) / np.max(x) * 20 + 20
py = np.ones_like(x) * 20 
pz = 20

t = np.zeros_like(x)
t = t + 0.2

model_abs = gyroid_utils.gyroid.GyroidModel(x, y, z, px, py, pz, t)

model_abs.compute_field(mode = "abs")

gyroid_utils.viz.twod_view_of_matrix(model_abs.v, model_abs.x, model_abs.y, model_abs.z,0,0.01)

model_abs.generate_mesh()
model_abs.simplify_mesh(target_faces = 10000)
model_abs.check_mesh_quality()

model_abs.save_mesh_preview(file_name + "-preview")

model_abs.save(file_name)

model_abs.export_stl(file_name)

In [ ]:
# =============================================
# ============== signed gyroids ==================
# =============================================
file_name = "gyroid-test-signed"

# ---  Define Period of gyroid unit cell -------
px = np.zeros_like(x) + 100
py = np.zeros_like(y) + 100
pz = np.zeros_like(z) + 100

t = np.zeros_like(x)
t = t + 0.5

model_signed = gyroid_utils.gyroid.GyroidModel(x, y, z, px, py, pz, t)

model_signed.compute_field(mode = "signed")

gyroid_utils.viz.twod_view_of_matrix(model_signed.v, model_signed.x, model_signed.y, model_signed.z,0,0.01)

model_signed.generate_mesh()
model_signed.simplify_mesh(target_faces = 10000)
model_signed.check_mesh_quality()
model_signed.save_mesh_preview(file_name + "-preview")

model_signed.save(file_name)

model_signed.export_stl(file_name)


In [ ]:
# =============================================
# ============== distance gyroids ==================
# =============================================
file_name = "gyroid-test-dist"

# ---  Define Period of gyroid unit cell -------
px = np.zeros_like(x) + 100
py = np.zeros_like(y) + 100
pz = np.zeros_like(z) + 100

t = np.zeros_like(x)
t = t + 4

model_dist = gyroid_utils.gyroid.GyroidModel(x, y, z, px, py, pz, t)

model_dist.compute_field(mode = "distance")

gyroid_utils.viz.twod_view_of_matrix(model_dist.v, model_dist.x, model_dist.y, model_dist.z,0,0.01)

model_dist.generate_mesh()
model_dist.simplify_mesh(target_faces = 10000)
model_dist.check_mesh_quality()
model_dist.save_mesh_preview(file_name + "-preview")

model_dist.save(file_name)

model_dist.export_stl(file_name)

## 3 — Reload a Saved Model

Demonstrates round-trip persistence: a previously saved `.npz` file is reloaded with the `GyroidModel.load()` class method, reconstructing the full model object (grid, scalar field, and mesh) without re-running any computation.

A 2-D slice of the reloaded scalar field is shown to confirm the data loaded correctly and matches the original.

In [ ]:
model_abs_2 = gyroid_utils.gyroid.GyroidModel.load(file_name + ".npz")
    
gyroid_utils.viz.twod_view_of_matrix(model_abs_2.v, model_abs_2.x, model_abs_2.y, model_abs_2.z,0,0.01)